# Portfolio Risk Analysis: Dhaka Stock Exchange, Extended Analysis
Munzeleen Sarwar

Hypothetical portfolio: Tk 1,000,000, equal weight across 11 DSE stocks (BATBC, BXPHARMA, CITYBANK, DBH, EBL, ROBI, ACI, BEXIMCO, FUWANGCER, GEMINISEA, SEAPEARL). Window: 3 Nov 2024 to 15 Oct 2026, 469 trading days. This notebook recomputes the headline extended analysis numbers from the saved data of the main simulation and checks them against data/extended_results.json. Seed 42 wherever randomness is used.

In [1]:
import json
import numpy as np
import pandas as pd
from scipy import stats

TICKERS = ['BATBC','BXPHARMA','CITYBANK','DBH','EBL','ROBI','ACI',
           'BEXIMCO','FUWANGCER','GEMINISEA','SEAPEARL']
frames = {}
for t in TICKERS:
    df = pd.read_csv(f'data/{t}.csv', parse_dates=['date']).set_index('date')
    frames[t] = df
closes = pd.DataFrame({t: frames[t]['close'] for t in TICKERS}).sort_index()
volumes = pd.DataFrame({t: frames[t]['volume'] for t in TICKERS}).sort_index()
logret = np.log(closes / closes.shift(1)).dropna()
simple = closes.pct_change().dropna()
R = json.load(open('data/extended_results.json'))
print('closes:', closes.shape, closes.index.min().date(), 'to', closes.index.max().date())

closes: (469, 11) 2024-11-03 to 2026-10-15


## 1. Risk decomposition
Component contributions from the covariance matrix of daily log returns.

In [2]:
w = np.full(11, 1/11)
cov_a = logret.cov().values * 252
port_vol = float(np.sqrt(w @ cov_a @ w))
crc = w * (cov_a @ w) / port_vol
share = pd.Series(100 * crc / port_vol, index=TICKERS).sort_values(ascending=False)
print('Portfolio volatility (% a year):', round(100 * port_vol, 2))
print(share.round(2).to_string())
print('Saved value:', R['risk_decomposition']['portfolio_volatility_pct'])

Portfolio volatility (% a year): 17.36
SEAPEARL     13.74
FUWANGCER    13.16
GEMINISEA    12.99
BXPHARMA     10.29
CITYBANK      8.22
DBH           7.99
ACI           7.90
EBL           7.14
ROBI          7.12
BATBC         7.09
BEXIMCO       4.36
Saved value: 17.36


## 2. Diversification

In [3]:
indiv = np.sqrt(np.diag(cov_a))
wavg = float((w * indiv).sum())
corr = logret.corr().values
iu = np.triu_indices(11, 1)
print('Weighted average individual volatility (%):', round(100 * wavg, 2))
print('Diversification ratio:', round(wavg / port_vol, 2))
print('Average pairwise correlation:', round(float(corr[iu].mean()), 3))

Weighted average individual volatility (%): 31.74
Diversification ratio: 1.83
Average pairwise correlation: 0.236


## 3. Drawdown and downside
Path definition: daily rebalanced equal weight portfolio of simple returns.

In [4]:
port = simple.mean(axis=1)
wealth = 1_000_000 * (1 + port).cumprod()
dd = wealth / wealth.cummax() - 1
down = np.sqrt((np.minimum(port, 0) ** 2).mean()) * np.sqrt(252)
print('Total return (%):', round(100 * float(wealth.iloc[-1] / 1e6 - 1), 2))
print('Maximum drawdown (%):', round(100 * float(dd.min()), 2), 'on', dd.idxmin().date())
print('Sortino ratio:', round(float(port.mean() * 252 / down), 3))
print('Daily VaR95 (%):', round(-100 * float(port.quantile(0.05)), 2))

Total return (%): -6.59
Maximum drawdown (%): -23.94 on 2026-09-14
Sortino ratio: -0.173
Daily VaR95 (%): 1.66


## 4. Distribution tests

In [5]:
port_log = logret.mean(axis=1)
jb = stats.jarque_bera(port_log.values)
print('Skewness:', round(float(stats.skew(port_log.values)), 3))
print('Excess kurtosis:', round(float(stats.kurtosis(port_log.values)), 3))
print('Jarque-Bera:', round(float(jb.statistic), 1), 'p-value:', float(jb.pvalue))

Skewness: -0.415
Excess kurtosis: 2.957
Jarque-Bera: 184.0 p-value: 1.126078583073632e-40


## 5. Liquidity: BEXIMCO check

In [6]:
traded = volumes['BEXIMCO'] * closes['BEXIMCO']
med = float(traded.median())
print('BEXIMCO median daily traded value (Tk):', med)
print('Median daily volume (shares):', float(volumes['BEXIMCO'].median()))
print('Days to exit Tk 90,909 at 20% of median:', round(90909.09 / (0.2 * med), 1))
liq = pd.read_csv('data/extended_liquidity.csv')
print(liq[['ticker', 'liquidity_grade']].to_string(index=False))

BEXIMCO median daily traded value (Tk): 330.29999999999995
Median daily volume (shares): 3.0
Days to exit Tk 90,909 at 20% of median: 1376.2
   ticker liquidity_grade
    BATBC            Good
 BXPHARMA            Good
 CITYBANK            Thin
      DBH            Good
      EBL            Thin
     ROBI            Thin
      ACI            Good
  BEXIMCO         Trapped
FUWANGCER            Thin
GEMINISEA            Good
 SEAPEARL            Good


## 6. Rolling volatility, EWMA and stress test check

In [7]:
roll = port_log.rolling(63).std(ddof=1) * np.sqrt(252)
print('Rolling 63d vol range (%):', round(100 * float(roll.min()), 2), 'to', round(100 * float(roll.max()), 2))
var = np.empty(len(port_log)); var[0] = float(port_log.var(ddof=1))
r2 = port_log.values ** 2
for i in range(1, len(port_log)):
    var[i] = 0.94 * var[i-1] + 0.06 * r2[i-1]
print('EWMA latest vol (% a year):', round(100 * float(np.sqrt(var[-1] * 252)), 2))
worst5 = port.nsmallest(5).values
print('Stress A, five worst days back to back, loss (Tk):', round(-(float(np.prod(1 + worst5)) - 1) * 1e6))

Rolling 63d vol range (%): 8.07 to 25.73
EWMA latest vol (% a year): 15.96
Stress A, five worst days back to back, loss (Tk): 190448


## 7. Fat tail Monte Carlo (Student-t), 252 days
Same mean and covariance basis as the GBM simulation, seed 42, 10,000 paths. Degrees of freedom from portfolio excess kurtosis by the method of moments.

In [8]:
excess = float(stats.kurtosis(port_log.values))
df_use = min(max(4 + 6 / excess, 3.5), 30.0)
print('Degrees of freedom used:', round(df_use, 2))
mu = logret.mean().values
cov_d = logret.cov().values
chol = np.linalg.cholesky(cov_d)
drift = mu - 0.5 * np.diag(cov_d)
rng = np.random.default_rng(42)
scale = np.sqrt((df_use - 2) / df_use)
endings = np.empty(10_000)
done = 0
while done < 10_000:
    b = min(1000, 10_000 - done)
    z = rng.standard_normal((b, 252, 11))
    chi = rng.chisquare(df_use, size=(b, 252, 1))
    shocks = (z @ chol.T) / np.sqrt(chi / df_use) * scale
    growth = np.exp(np.cumsum(drift + shocks, axis=1))
    endings[done:done + b] = 1_000_000 * growth.mean(axis=2)[:, -1]
    done += b
rets = endings / 1e6 - 1
print('Median ending value (Tk):', round(float(np.median(endings))))
print('VaR95 (Tk):', round(-float(np.percentile(rets, 5)) * 1e6))
print('Saved:', R['student_t']['results']['252']['median_value_tk'],
      R['student_t']['results']['252']['var95_tk'])

Degrees of freedom used: 6.03


Median ending value (Tk): 954377
VaR95 (Tk): 284293
Saved: 954377.2685261418 284293.0


## 8. Beta against DSEX and fundamentals
Beta status comes from data/extended_results.json. Fundamental figures are read from data/extended_fundamentals.csv.

In [9]:
print('Beta status:', R['beta_vs_dsex']['status'])
print(R['beta_vs_dsex']['reason'][:300])
fund = pd.read_csv('data/extended_fundamentals.csv')
print(fund[['ticker', 'eps_ttm_bdt', 'pe_ratio', 'dividend_yield_pct']].to_string(index=False))
print('Loss making stocks:', fund.loc[fund['eps_ttm_bdt'] < 0, 'ticker'].tolist())

Beta status: not_computed
No daily DSEX index series for 3 Nov 2024 to 15 Oct 2026 could be fetched from any source tried. The DSE day-end API used for the stock prices holds no index instruments (DSEX, DSEX_IDX, DSE30, DSES and INDEX all returned zero rows). stocksurferbd index history reads old.dsebd.org, which did not res
   ticker  eps_ttm_bdt  pe_ratio  dividend_yield_pct
    BATBC         6.98     29.97                1.44
 BXPHARMA        19.39      7.41                3.31
 CITYBANK         8.86      3.36                4.38
      DBH         5.01      7.77                3.86
      EBL         5.58      4.25               10.55
     ROBI         2.00     14.82                5.89
      ACI        -2.48       NaN                1.41
  BEXIMCO        -4.21       NaN                 NaN
FUWANGCER        -0.04       NaN                0.89
GEMINISEA        -7.41       NaN                0.70
 SEAPEARL        -2.23       NaN                0.15
Loss making stocks: ['ACI', 'BEXIMCO'